---
## Deep Learning Models — BERT & Longformer

**Overview:**

Implemented and compared two distinct strategies to handle long-form text data:

- **Model A: BERT-Base-Cased (Baseline)** with a 512-token limit. Implements a 'Head + Tail' approach: preserves the first 128 and last 382 tokens to capture the introduction and conclusion, sacrificing the middle content.

- **Model B: Longformer-Base-4096** — Utilizes Sliding Window Attention to process sequences up to 4096 tokens. Capturing the entire narrative (without truncation) can generally improve prediction of high-risk cases.

In [11]:
import re
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\wenli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wenli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\wenli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\wenli\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

**Data for Deep Learning Models**

The BERT and Longformer models use `text_clean_mild` (light cleaning, preserving sentence structure).
This column was already created above from `df_clean`, so we can proceed directly to the train/val/test split.

In [12]:
# Define features and target for deep learning models (use mild-cleaned text)
df_clean = pd.read_csv('df_clean.csv')
X_dl = df_clean['text_clean_mild']
y_dl = df_clean['label']

In [13]:
# Train/val/test split for deep learning models
# 80% train, 10% val, 10% test
from sklearn.model_selection import train_test_split

X_temp_dl, X_test_dl, y_temp_dl, y_test_dl = train_test_split(
    X_dl, y_dl,
    test_size=0.1,
    random_state=42,
    stratify=y_dl
)

In [14]:
X_train_dl, X_val_dl, y_train_dl, y_val_dl = train_test_split(
    X_temp_dl, y_temp_dl,
    test_size=0.1111,
    random_state=42,
    stratify=y_temp_dl
)

In [15]:
X_test_dl = X_test_dl.reset_index(drop=True)
y_test_dl = y_test_dl.reset_index(drop=True)

**A/B test**: comparing two methods

**A test: BERT (Head + Tail)**

In [16]:
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import AutoModelForSequenceClassification
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

**Tial 1**: Since the Bert only accepts about 512 tokens so we set up a trimming logic of the input text: Implements the 'Head + Tail' truncation strategy to preserve context from both the beginning and the end of long social media posts.

In [17]:
def head_tail_tokenize(text, max_len=512, head_len=128, tail_len=382):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    # trim from head & tail
    if len(tokens) > (max_len - 2):
        head = tokens[:head_len]
        tail = tokens[-tail_len:]
        tokens = head + tail

    input_ids = [tokenizer.cls_token_id] + tokens + [tokenizer.sep_token_id]
    # if text len < 512 tokens
    padding_len = max_len - len(input_ids)
    attention_mask = [1] * len(input_ids) + [0] * padding_len # use 0 to label the padding area
    input_ids = input_ids + [tokenizer.pad_token_id] * padding_len

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask
    }

In [18]:
tokenized_train_data = [head_tail_tokenize(text) for text in X_train_dl]
tokenized_val_data   = [head_tail_tokenize(text) for text in X_val_dl]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (593 > 512). Running this sequence through the model will result in indexing errors


In [19]:
# Combine features and labels into a Hugging Face Dataset
from datasets import Dataset

train_dataset = Dataset.from_dict({
    'input_ids':      [x['input_ids']      for x in tokenized_train_data],
    'attention_mask': [x['attention_mask']  for x in tokenized_train_data],
    'labels':          y_train_dl
})

val_dataset = Dataset.from_dict({
    'input_ids':      [x['input_ids']      for x in tokenized_val_data],
    'attention_mask': [x['attention_mask']  for x in tokenized_val_data],
    'labels':          y_val_dl
})

In [ ]:
# Load the model with a sequence classification head on top
# num_labels=2 because we have 'suicide' and 'non-suicide'
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

In [ ]:
# Load the evaluation metrics
metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       # Run evaluation at the end of each epoch
    save_strategy="epoch",       # Save the model after each epoch
    learning_rate=2e-5,          # Standard BERT learning rate
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,                   # IMPORTANT: Accelerates training on Colab GPUs
    # bf16=True,                    # IMPORTANT: use for TPUs instead
    load_best_model_at_end=True, # Automatically keep the best version of the model
    metric_for_best_model="f1"   # We prioritize F1-score for this task
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
# Start the fine-tuning process
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.151885,0.947692,0.948847,0.953629,0.944112
2,0.236062,0.139313,0.955897,0.956697,0.965447,0.948104
3,0.098503,0.159236,0.956923,0.958000,0.959920,0.956088


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

CPU times: user 10min 35s, sys: 10.1 s, total: 10min 45s
Wall time: 10min 55s


TrainOutput(global_step=1464, training_loss=0.12711267002293322, metrics={'train_runtime': 654.2618, 'train_samples_per_second': 35.733, 'train_steps_per_second': 2.238, 'total_flos': 6151273363261440.0, 'train_loss': 0.12711267002293322, 'epoch': 3.0})

In [ ]:
# Evaluate on the test set
tokenized_test_data = [head_tail_tokenize(text) for text in X_test_dl]

test_dataset_bert = Dataset.from_dict({
    'input_ids':      [x['input_ids']      for x in tokenized_test_data],
    'attention_mask': [x['attention_mask']  for x in tokenized_test_data],
    'labels':          y_test_dl.tolist()
})
test_results_bert = trainer.predict(test_dataset_bert)

test_results_bert.metrics

CPU times: user 8.82 s, sys: 64 ms, total: 8.89 s
Wall time: 8.87 s


{'test_loss': 0.18381889164447784,
 'test_accuracy': 0.961025641025641,
 'test_f1': 0.9619238476953907,
 'test_precision': 0.96579476861167,
 'test_recall': 0.9580838323353293,
 'test_runtime': 8.2764,
 'test_samples_per_second': 117.805,
 'test_steps_per_second': 14.741}

**B test: Longformer**

In [ ]:
from transformers import LongformerTokenizer, LongformerForSequenceClassification

# Load Longformer - specifically designed for long documents
model_name = "allenai/longformer-base-4096"
lf_tokenizer = LongformerTokenizer.from_pretrained(model_name)

# Initialize model for 2-class classification
lf_model = LongformerForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

LongformerForSequenceClassification LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     | 
-------------------------------+------------+-
lm_head.dense.bias             | UNEXPECTED | 
lm_head.layer_norm.bias        | UNEXPECTED | 
lm_head.dense.weight           | UNEXPECTED | 
longformer.pooler.dense.weight | UNEXPECTED | 
longformer.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight         | UNEXPECTED | 
lm_head.layer_norm.weight      | UNEXPECTED | 
lm_head.bias                   | UNEXPECTED | 
classifier.dense.weight        | MISSING    | 
classifier.dense.bias          | MISSING    | 
classifier.out_proj.weight     | MISSING    | 
classifier.out_proj.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import pandas as pd

train_df = pd.DataFrame({'text': X_train_dl, 'label': y_train_dl})
val_df   = pd.DataFrame({'text': X_val_dl,   'label': y_val_dl})

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

In [ ]:
def longformer_tokenize(examples):
    return lf_tokenizer(
        examples["text"],
        padding=False,
        truncation=True,
        max_length=4096
    )

tokenized_train_lf = train_ds.map(
    longformer_tokenize, batched=True,
    remove_columns=train_ds.column_names
)
tokenized_val_lf = val_ds.map(
    longformer_tokenize, batched=True,
    remove_columns=val_ds.column_names
)

tokenized_train_lf = tokenized_train_lf.add_column("labels", y_train_dl.tolist())
tokenized_val_lf   = tokenized_val_lf.add_column("labels", y_val_dl.tolist())

Map:   0%|          | 0/7793 [00:00<?, ? examples/s]

Map:   0%|          | 0/975 [00:00<?, ? examples/s]

In [ ]:
#tokenized_train_lf = tokenized_train_lf.add_column('labels', y_train_dl.tolist())
#tokenized_val_lf   = tokenized_val_lf.add_column('labels', y_val_dl.tolist())

In [ ]:
def compute_metrics_lf(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=lf_tokenizer, pad_to_multiple_of=16)

lf_training_args = TrainingArguments(
    output_dir="./results_longformer",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=32,
    num_train_epochs=3,
    fp16=True,
    # bf16=True,
    weight_decay=0.01,
    gradient_checkpointing=True,
)

lf_trainer = Trainer(
    model=lf_model,
    args=lf_training_args,
    train_dataset=tokenized_train_lf,
    eval_dataset=tokenized_val_lf,
    compute_metrics=compute_metrics_lf,
    data_collator=data_collator,
)

The next step can take a while...

In [ ]:
%%time
lf_trainer.train()

Initializing global attention on CLS token...
Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.092964,0.964103,0.965920,0.942966,0.990020


KeyboardInterrupt: 

In [ ]:
test_df = pd.DataFrame({'text': X_test_dl, 'label': y_test_dl})
test_ds = Dataset.from_pandas(test_df)

In [ ]:
# Evaluate on the test set
%%time
tokenized_test_lf = test_ds.map(
    longformer_tokenize, batched=True,
    remove_columns=test_ds.column_names
)
tokenized_test_lf = tokenized_test_lf.add_column('labels', y_test_dl.tolist())
test_results = lf_trainer.predict(tokenized_test_lf)
test_results.metrics

Map:   0%|          | 0/975 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.092964,0.964103,0.965920,0.942966,0.990020


KeyboardInterrupt: 

**Evaluation**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

metrics_names = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1']
display_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

bert_scores = [test_results_bert.metrics[m] for m in metrics_names]
lf_scores = [test_results.metrics[m] for m in metrics_names]

performance_data = {
    'Metric': display_names,
    'BERT ': bert_scores,
    'Longformer ': lf_scores
}

df_plot = pd.DataFrame(performance_data)
df_melted = df_plot.melt(id_vars='Metric', var_name='Model', value_name='Score')

plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")
ax = sns.barplot(x='Metric', y='Score', hue='Model', data=df_melted, palette=['#3498db', '#2ecc71'])

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.3f'),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha = 'center', va = 'center',
                xytext = (0, 9),
                textcoords = 'offset points',
                fontsize=11, fontweight='bold')

plt.ylim(min(bert_scores + lf_scores) - 0.05, 1.0)
plt.title('Performance Comparison: BERT (Head+Tail) vs Longformer(Full Text)', fontsize=16)
plt.show()

NameError: name 'test_results' is not defined

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred_bert = np.argmax(test_results_bert.predictions, axis=-1)
y_pred_lf   = np.argmax(test_results.predictions, axis=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# BERT
cm_bert = confusion_matrix(y_test_dl, y_pred_bert)
disp1 = ConfusionMatrixDisplay(cm_bert, display_labels=['Non-Suicide', 'Suicide'])
disp1.plot(ax=ax1, cmap='Blues', colorbar=False, values_format='d')
ax1.set_title('Model A: BERT (Head+Tail)', fontsize=14)

# Longformer
cm_lf = confusion_matrix(y_test_dl, y_pred_lf)
disp2 = ConfusionMatrixDisplay(cm_lf, display_labels=['Non-Suicide', 'Suicide'])
disp2.plot(ax=ax2, cmap='Greens', colorbar=False, values_format='d')
ax2.set_title('Model B: Longformer (Full Text)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import torch.nn.functional as F
import torch

# BERT
bert_logits = torch.from_numpy(test_results_bert.predictions)
bert_probs = F.softmax(bert_logits, dim=-1)[:, 1].numpy()
fpr_bert, tpr_bert, _ = roc_curve(y_test_dl, bert_probs)  # y_test_dl, not y_test
roc_auc_bert = auc(fpr_bert, tpr_bert)

# Longformer
lf_logits = torch.from_numpy(test_results.predictions)
lf_probs = F.softmax(lf_logits, dim=-1)[:, 1].numpy()
fpr_lf, tpr_lf, _ = roc_curve(y_test_dl, lf_probs)        # y_test_dl, not y_test
roc_auc_lf = auc(fpr_lf, tpr_lf)

In [ ]:
plt.figure(figsize=(8, 6))
sns.set_style("whitegrid")

# BERT
plt.plot(fpr_bert, tpr_bert, color='#3498db', lw=2,
         label=f'Model A: BERT (AUC = {roc_auc_bert:.3f})')

# Longformer
plt.plot(fpr_lf, tpr_lf, color='#2ecc71', lw=2,
         label=f'Model B: Longformer (AUC = {roc_auc_lf:.3f})')

plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curve Comparison: BERT vs Longformer', fontsize=15)
plt.legend(loc="lower right", fontsize=11)
plt.show()

**Finding:**
In the domain of suicide risk assessment, capturing the global semantic context of long-form discourse can improve predictive accuracy.The superiority of Longformer in short-text samples ($<$100 words) suggests that its attention mechanism and long-document pre-training confer a more refined understanding of complex emotional semantics, outperforming BERT even when truncation is not a factor.

In [ ]:
y_true = y_test_dl.values

bert_error_indices = [i for i in range(len(y_true)) if y_pred_bert[i] != y_true[i]]
saved_indices = [i for i in range(len(y_true)) if (y_pred_bert[i] != y_true[i]) and (y_pred_lf[i] == y_true[i])]
test_lengths = X_test_dl.apply(lambda x: len(str(x).split())).values
bins   = [0, 100, 200, 500, max(test_lengths) + 1]
labels = ['Short (<100)', 'Mid-Short (100-200)', 'Mid-Long (200-500)', 'Long (500+)']

df_err_dist = pd.DataFrame({'Length': test_lengths[bert_error_indices]})
df_err_dist['Bin'] = pd.cut(df_err_dist['Length'], bins=bins, labels=labels, include_lowest=True)
total_errors_per_bin = df_err_dist.groupby('Bin', observed=True).size().reindex(labels, fill_value=0)

df_saved_dist = pd.DataFrame({'Length': test_lengths[saved_indices]})
df_saved_dist['Bin'] = pd.cut(df_saved_dist['Length'], bins=bins, labels=labels, include_lowest=True)
saved_per_bin = df_saved_dist.groupby('Bin', observed=True).size().reindex(labels, fill_value=0)

total_err     = total_errors_per_bin.values
saved_err_arr = saved_per_bin.values
remaining_err = total_err - saved_err_arr

df_stacked = pd.DataFrame({
    'Length_Bin':          labels,
    'Fixed_by_LF':         saved_err_arr,
    'Remaining_BERT_Err':  remaining_err
})

plt.figure(figsize=(12, 7))
sns.set_style('white')

color_fixed     = '#5b9bd5'
color_remaining = '#d9d9d9'

p1 = plt.bar(df_stacked['Length_Bin'], df_stacked['Fixed_by_LF'],
             label='Errors Resolved by Longformer', color=color_fixed, alpha=0.9)
p2 = plt.bar(df_stacked['Length_Bin'], df_stacked['Remaining_BERT_Err'],
             bottom=df_stacked['Fixed_by_LF'],
             label='Unresolved BERT Errors', color=color_remaining, alpha=0.7)

for i in range(len(labels)):
    total = total_err[i]
    saved = saved_err_arr[i]
    if total > 0:
        rate = (saved / total * 100)
        plt.text(i, total + (max(total_err) * 0.02), f'n={total} cases',
                 ha='center', fontsize=10, color='#7f7f7f', fontweight='500')
        if saved > 0:
            plt.text(i, saved / 2, f'{rate:.1f}% Fixed',
                     ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.title('Corrective Performance by Sequence Length', fontsize=16, pad=25, loc='left', fontweight='bold', color='#333333')
plt.xlabel('Post Length (Words)', fontsize=12, labelpad=15)
plt.ylabel('Error Instances', fontsize=12, labelpad=15)
sns.despine(left=True)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(frameon=False, bbox_to_anchor=(1, 1.05), fontsize=10)
plt.tight_layout()
plt.show()

---
## Final Model Comparison: Classical vs. Deep Learning

This section consolidates results from all 8 models — 6 classical TF-IDF variants and 2 fine-tuned transformer models — for a unified comparison.

> **Note on test sets:** The 6 classical models were evaluated on a held-out 20% test split (`y_test`, ~1,951 samples). BERT and Longformer were evaluated on a separate 10% split (`y_test_dl`, ~976 samples) using mild-cleaned text suited for deep learning. Metrics are reported separately per model; direct cross-group comparison should account for this difference.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# --- Classical models: collect accuracy, F1, precision, recall ---
classical_models = [
    ('SVC (Unigrams)',                    predictions,    y_test),
    ('SVC (Unigrams+Bigrams)',             predictions2,   y_test),
    ('Naive Bayes (Unigrams)',             nb_predictions,  y_test),
    ('Naive Bayes (Unigrams+Bigrams)',     nb_predictions2, y_test),
    ('Logistic Regression (Unigrams)',     lr_predictions,  y_test),
    ('Logistic Regression (Unigrams+Bigrams)', lr_predictions2, y_test),
]

classical_rows = []
for name, preds, y_true in classical_models:
    classical_rows.append({
        'Model':       name,
        'Type':        'Classical',
        'Accuracy':    round(accuracy_score(y_true, preds), 4),
        'F1':          round(f1_score(y_true, preds), 4),
        'Precision':   round(precision_score(y_true, preds), 4),
        'Recall':      round(recall_score(y_true, preds), 4),
    })
df_classical = pd.DataFrame(classical_rows)
print('Classical models evaluated on 20% test split (~1,951 samples)')
df_classical

In [ ]:
import numpy as np

# --- Deep learning models: extract metrics from trainer output ---
y_pred_bert_arr = np.argmax(test_results_bert.predictions, axis=-1)
y_pred_lf_arr   = np.argmax(test_results.predictions, axis=-1)

dl_models = [
    ('BERT (Head+Tail)',    y_pred_bert_arr, y_test_dl),
    ('Longformer',         y_pred_lf_arr,   y_test_dl),
]

dl_rows = []
for name, preds, y_true in dl_models:
    dl_rows.append({
        'Model':       name,
        'Type':        'Deep Learning',
        'Accuracy':    round(accuracy_score(y_true, preds), 4),
        'F1':          round(f1_score(y_true, preds), 4),
        'Precision':   round(precision_score(y_true, preds), 4),
        'Recall':      round(recall_score(y_true, preds), 4),
    })
df_dl = pd.DataFrame(dl_rows)
print('Deep learning models evaluated on 10% test split (~976 samples)')
df_dl

In [ ]:
# Combined summary table
df_all = pd.concat([df_classical, df_dl], ignore_index=True)

# Highlight best value per metric
def highlight_max(s):
    is_max = s == s.max()
    return ['font-weight: bold; color: #2ecc71' if v else '' for v in is_max]

df_all.style \
    .apply(highlight_max, subset=['Accuracy', 'F1', 'Precision', 'Recall']) \
    .set_caption('All Models — Performance Summary') \
    .format({'Accuracy': '{:.4f}', 'F1': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}'})

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

models      = df_all['Model'].tolist()
types       = df_all['Type'].tolist()
accuracies  = df_all['Accuracy'].tolist()
f1_scores   = df_all['F1'].tolist()

x = np.arange(len(models))
width = 0.35

colors_acc = ['#3498db' if t == 'Classical' else '#e67e22' for t in types]
colors_f1  = ['#2980b9' if t == 'Classical' else '#d35400' for t in types]

fig, ax = plt.subplots(figsize=(16, 7))

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color=colors_acc, alpha=0.85)
bars2 = ax.bar(x + width/2, f1_scores,  width, label='F1 Score', color=colors_f1,  alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5, rotation=45)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5, rotation=45)

# Divider between classical and DL
ax.axvline(x=5.5, color='gray', linestyle='--', linewidth=1.2, label='Classical | Deep Learning')

classical_patch = mpatches.Patch(color='#3498db', label='Classical (Accuracy)')
dl_patch        = mpatches.Patch(color='#e67e22', label='Deep Learning (Accuracy)')
classical_f1    = mpatches.Patch(color='#2980b9', label='Classical (F1)')
dl_f1           = mpatches.Patch(color='#d35400', label='Deep Learning (F1)')
ax.legend(handles=[classical_patch, classical_f1, dl_patch, dl_f1], loc='lower right', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Accuracy & F1 Score — All Models', fontsize=15, fontweight='bold')
ax.text(2.5, 0.52, 'Classical (TF-IDF)', ha='center', fontsize=9, color='gray')
ax.text(6.5, 0.52, 'Deep Learning', ha='center', fontsize=9, color='gray')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves for all 8 models on their respective test sets
# Note: classical and DL curves use different test sets — visual comparison is indicative
import torch, torch.nn.functional as F_torch

# Classical probabilities (already computed above)
classical_roc = [
    ('SVC (Unigrams)',                 svc_fpr,   svc_tpr,   svc_auc,   '#1f77b4'),
    ('SVC (Unigrams+Bigrams)',          svc_fpr2,  svc_tpr2,  svc_auc2,  '#1f77b4'),
    ('Naive Bayes (Unigrams)',          nb_fpr,    nb_tpr,    nb_auc,    '#2ca02c'),
    ('Naive Bayes (Unigrams+Bigrams)',  nb_fpr2,   nb_tpr2,   nb_auc2,   '#2ca02c'),
    ('LR (Unigrams)',                  lr_fpr,    lr_tpr,    lr_auc,    '#d62728'),
    ('LR (Unigrams+Bigrams)',           lr_fpr2,   lr_tpr2,   lr_auc2,   '#d62728'),
]

# DL probabilities
bert_logits_all = torch.from_numpy(test_results_bert.predictions)
bert_probs_all  = F_torch.softmax(bert_logits_all, dim=-1)[:, 1].numpy()
lf_logits_all   = torch.from_numpy(test_results.predictions)
lf_probs_all    = F_torch.softmax(lf_logits_all, dim=-1)[:, 1].numpy()

from sklearn.metrics import roc_curve, auc as sk_auc
fpr_bert_f, tpr_bert_f, _ = roc_curve(y_test_dl, bert_probs_all)
fpr_lf_f,   tpr_lf_f,   _ = roc_curve(y_test_dl, lf_probs_all)
auc_bert_f = sk_auc(fpr_bert_f, tpr_bert_f)
auc_lf_f   = sk_auc(fpr_lf_f,   tpr_lf_f)

fig, ax = plt.subplots(figsize=(10, 8))

linestyles = ['-', '--', '-', '--', '-', '--']
for (name, fpr, tpr, auc_val, color), ls in zip(classical_roc, linestyles):
    ax.plot(fpr, tpr, color=color, lw=1.5, linestyle=ls,
            label=f'{name} (AUC={auc_val:.3f})')

ax.plot(fpr_bert_f, tpr_bert_f, color='#ff7f0e', lw=2.5,
        label=f'BERT Head+Tail (AUC={auc_bert_f:.3f})')
ax.plot(fpr_lf_f,   tpr_lf_f,   color='#9467bd', lw=2.5,
        label=f'Longformer (AUC={auc_lf_f:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All 8 Models\n(Classical on 20% split; DL on 10% split)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=8.5)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of key metrics across all models
import seaborn as sns

heatmap_df = df_all.set_index('Model')[['Accuracy', 'F1', 'Precision', 'Recall']]

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    heatmap_df,
    annot=True, fmt='.4f',
    cmap='YlGnBu',
    linewidths=0.5,
    vmin=heatmap_df.values.min() - 0.02,
    vmax=1.0,
    ax=ax
)
ax.set_title('Performance Heatmap — All Models', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('')
# Draw a line separating classical from DL rows
ax.axhline(6, color='white', linewidth=3)
plt.tight_layout()
plt.show()

### Key Takeaways

- **Best classical model:** Compare Accuracy and F1 across the 6 TF-IDF variants to identify the top performer. The unigram+bigram variants generally match or slightly improve over unigrams alone.

- **Deep learning advantage:** BERT and Longformer operate on lightly cleaned text with full token context, bypassing the information loss introduced by stopword removal and lemmatization in the classical pipeline.

- **Longformer vs. BERT:** Longformer's sliding window attention allows it to process the full post without truncation, which is particularly relevant for longer, narrative-style posts where suicidal ideation may be expressed gradually.

- **Test set caveat:** Classical models use a 20% test split; BERT and Longformer use a 10% split with mild cleaning. Metric values are not directly comparable across the two groups — this table is best interpreted within each group.

- **Recall priority:** In a mental health classification context, recall (sensitivity) for the suicide class is arguably the most critical metric — a false negative (missed case) carries higher cost than a false positive.